In [23]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [24]:
!pip install -q torchmetrics mlflow tqdm

In [25]:
import torch
import shutil, os, glob
import mlflow
import pandas as pd
import sys
import argparse
from dotenv import load_dotenv
from pathlib import Path

In [26]:
load_dotenv()

True

In [27]:
# Check GPU
if torch.cuda.is_available():
    print(f"CUDA available: {torch.cuda.is_available()}")
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("WARNING: No GPU")

CUDA available: True
GPU: Tesla T4
VRAM: 15.6 GB


In [28]:
# Set Drive path
DRIVE_PATH = Path(os.environ["DRIVE_PATH"])
os.makedirs(f'{DRIVE_PATH}/checkpoints', exist_ok=True)
os.makedirs(f'{DRIVE_PATH}/data/splits', exist_ok=True)
os.makedirs(f'{DRIVE_PATH}/data/labeled', exist_ok=True)
print(f"Drive path ready: {DRIVE_PATH}")

Drive path ready: /content/drive/MyDrive/Colab Notebooks/ad-creative-analyzer


In [29]:
if str(DRIVE_PATH) not in sys.path:
    sys.path.append(str(DRIVE_PATH))

from src.train import main as train_main

In [30]:
# Set MLflow tracking URI to local SQLite database
mlflow.set_tracking_uri(f"sqlite:///{DRIVE_PATH}/mlflow.db")
mlflow.set_experiment("visiomark-image-model")
print("MLflow ready")

MLflow ready


In [31]:
# Check data splits
splits = {}
for split in ['train', 'val', 'test']:
    path = f"{DRIVE_PATH}/data/splits/{split}.csv"
    if os.path.exists(path):
        df = pd.read_csv(path)
        splits[split] = df
        print(f"{split}: {len(df)} samples")
        print(f"content_type distribution: {df['content_type_label'].value_counts().to_dict()}")
        print(f"mood distribution:         {df['mood_label'].value_counts().to_dict()}")
        print()
    else:
        print(f"{split}.csv not found")

train: 1131 samples
content_type distribution: {1: 609, 2: 355, 0: 167}
mood distribution:         {1: 423, 2: 352, 0: 224, 3: 132}

val: 243 samples
content_type distribution: {1: 131, 2: 76, 0: 36}
mood distribution:         {1: 91, 2: 76, 0: 47, 3: 29}

test: 243 samples
content_type distribution: {1: 131, 2: 76, 0: 36}
mood distribution:         {1: 90, 2: 76, 0: 48, 3: 29}



### Phase 1

In [32]:
print("Starting Phase 1.")
print("=" * 60)

args_phase1 = argparse.Namespace(
    phase=1,
    epochs=10,
    batch_size=16,
    lr=1e-3,
    img_dir=f"{DRIVE_PATH}/data/raw/",
    train_csv=f"{DRIVE_PATH}/data/splits/train.csv",
    val_csv=f"{DRIVE_PATH}/data/splits/val.csv",
    checkpoint=None,
    device='cuda' if torch.cuda.is_available() else 'cpu',
    save_dir=f"{DRIVE_PATH}/checkpoints",
)

checkpoints_path = train_main(args_phase1)
print(f"Phase 1 checkpoints saved to: {checkpoints_path}")

Starting Phase 1.
Epoch 01/10 | loss=2.3279 | F1_content=0.5845 | F1_mood=0.4947


2026/06/07 13:22:55 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/06/07 13:22:55 WARNING mlflow.pytorch: Saving pytorch model by Pickle or CloudPickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is to set `serialization_format` to 'pt2' to save the PyTorch model using the safe graph model format.
2026/06/07 13:22:56 WARNING mlflow.utils.requirements_utils: Found torch version (2.11.0+cu128) contains a local version label (+cu128). MLflow logged a pip requirement for this package as 'torch==2.11.0' without the local version label to make it installable from PyPI. To specify pip requirements containing local version labels, please use `conda_env` or `pip_requirements`.
2026/06/07 13:23:06 WARNING mlflow.utils.requirements_utils: Found torchvision version (0.26.0+cu128) contains a local version

  ✓ Checkpoint saved (avg_F1=0.5396)


2026/06/07 13:23:29 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


Epoch 02/10 | loss=2.0292 | F1_content=0.6413 | F1_mood=0.5174


...

  ✓ Checkpoint saved (avg_F1=0.6222)
Epoch 09/10 | loss=1.6799 | F1_content=0.6788 | F1_mood=0.5601


2026/06/07 13:27:32 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


Epoch 10/10 | loss=1.6847 | F1_content=0.6727 | F1_mood=0.5772


2026/06/07 13:27:32 WARNING mlflow.pytorch: Saving pytorch model by Pickle or CloudPickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is to set `serialization_format` to 'pt2' to save the PyTorch model using the safe graph model format.
2026/06/07 13:27:32 WARNING mlflow.utils.requirements_utils: Found torch version (2.11.0+cu128) contains a local version label (+cu128). MLflow logged a pip requirement for this package as 'torch==2.11.0' without the local version label to make it installable from PyPI. To specify pip requirements containing local version labels, please use `conda_env` or `pip_requirements`.
2026/06/07 13:27:39 WARNING mlflow.utils.requirements_utils: Found torchvision version (0.26.0+cu128) contains a local version label (+cu128). MLflow logged a pip requirement for this package as 'torchvision==0.26.0' without the loca

  ✓ Checkpoint saved (avg_F1=0.6249)

✅ Phase 1 done. Best avg F1 = 0.6249
Phase 1 checkpoints saved to: /content/drive/MyDrive/Colab Notebooks/ad-creative-analyzer/checkpoints/best_phase1.pt


### Phase 2

In [33]:
print("Starting Phase 2.")
print("=" * 60)

phase1_checkpoints = sorted(glob.glob(f'{DRIVE_PATH}/checkpoints/best_phase1*.pt'))

if not phase1_checkpoints:
    raise FileNotFoundError("No Phase 1 checkpoint found.")

best_phase1 = phase1_checkpoints[-1]

args_phase2 = argparse.Namespace(
    phase=2,
    epochs=40,
    batch_size=16,
    lr=1e-5,
    img_dir=f"{DRIVE_PATH}/data/raw/",
    train_csv=f"{DRIVE_PATH}/data/splits/train.csv",
    val_csv=f"{DRIVE_PATH}/data/splits/val.csv",
    checkpoint=best_phase1,
    device='cuda' if torch.cuda.is_available() else 'cpu',
    save_dir=f"{DRIVE_PATH}/checkpoints",
)

final_checkpoints_path = train_main(args_phase2)
print(f"Phase 2 checkpoints saved to: {final_checkpoints_path}")

Starting Phase 2.
Loading Phase 1 checkpoint: /content/drive/MyDrive/Colab Notebooks/ad-creative-analyzer/checkpoints/best_phase1.pt
Loading Phase 1 checkpoint: /content/drive/MyDrive/Colab Notebooks/ad-creative-analyzer/checkpoints/best_phase1.pt
Unfroze last 3 encoder blocks
Epoch 01/40 | loss=1.6797 | F1_content=0.6773 | F1_mood=0.5781


2026/06/07 13:28:07 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/06/07 13:28:07 WARNING mlflow.pytorch: Saving pytorch model by Pickle or CloudPickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is to set `serialization_format` to 'pt2' to save the PyTorch model using the safe graph model format.
2026/06/07 13:28:07 WARNING mlflow.utils.requirements_utils: Found torch version (2.11.0+cu128) contains a local version label (+cu128). MLflow logged a pip requirement for this package as 'torch==2.11.0' without the local version label to make it installable from PyPI. To specify pip requirements containing local version labels, please use `conda_env` or `pip_requirements`.
2026/06/07 13:28:14 WARNING mlflow.utils.requirements_utils: Found torchvision version (0.26.0+cu128) contains a local version

  ✓ Checkpoint saved (avg_F1=0.6277)
Epoch 02/40 | loss=1.6544 | F1_content=0.6799 | F1_mood=0.5545


2026/06/07 13:29:05 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


Epoch 03/40 | loss=1.6076 | F1_content=0.6830 | F1_mood=0.5834


...

  ✓ Checkpoint saved (avg_F1=0.6618)


2026/06/07 13:41:18 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


Epoch 28/40 | loss=1.1372 | F1_content=0.7011 | F1_mood=0.6333


2026/06/07 13:41:19 WARNING mlflow.pytorch: Saving pytorch model by Pickle or CloudPickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is to set `serialization_format` to 'pt2' to save the PyTorch model using the safe graph model format.
2026/06/07 13:41:19 WARNING mlflow.utils.requirements_utils: Found torch version (2.11.0+cu128) contains a local version label (+cu128). MLflow logged a pip requirement for this package as 'torch==2.11.0' without the local version label to make it installable from PyPI. To specify pip requirements containing local version labels, please use `conda_env` or `pip_requirements`.
2026/06/07 13:41:26 WARNING mlflow.utils.requirements_utils: Found torchvision version (0.26.0+cu128) contains a local version label (+cu128). MLflow logged a pip requirement for this package as 'torchvision==0.26.0' without the loca

  ✓ Checkpoint saved (avg_F1=0.6672)
Epoch 29/40 | loss=1.2205 | F1_content=0.6958 | F1_mood=0.6077
Epoch 30/40 | loss=1.1876 | F1_content=0.6990 | F1_mood=0.6220
Epoch 31/40 | loss=1.1356 | F1_content=0.6954 | F1_mood=0.6233
Epoch 32/40 | loss=1.1809 | F1_content=0.7089 | F1_mood=0.6068
Epoch 33/40 | loss=1.1682 | F1_content=0.7024 | F1_mood=0.6294
Epoch 34/40 | loss=1.1524 | F1_content=0.7000 | F1_mood=0.6177
Epoch 35/40 | loss=1.1363 | F1_content=0.6856 | F1_mood=0.6098
Epoch 36/40 | loss=1.1487 | F1_content=0.7091 | F1_mood=0.6171
Epoch 37/40 | loss=1.1720 | F1_content=0.6992 | F1_mood=0.6249
Epoch 38/40 | loss=1.1491 | F1_content=0.6984 | F1_mood=0.6097
Epoch 39/40 | loss=1.1683 | F1_content=0.7067 | F1_mood=0.6021
Epoch 40/40 | loss=1.1760 | F1_content=0.6963 | F1_mood=0.6119

✅ Phase 2 done. Best avg F1 = 0.6672
Phase 2 checkpoints saved to: /content/drive/MyDrive/Colab Notebooks/ad-creative-analyzer/checkpoints/best_phase2.pt


In [34]:
# Copy MLflow DB if using local mode
db_source = f"{DRIVE_PATH}/mlflow.db"
db_backup = f"{DRIVE_PATH}/mlflow_backup.db"

if os.path.exists(db_source):
    shutil.copy2(db_source, db_backup)
    print(f"MLflow database backed up to: {db_backup}")
else:
    print("MLflow DB not found.")

print("\nALL TRAINING COMPLETE!")

MLflow database backed up to: /content/drive/MyDrive/Colab Notebooks/ad-creative-analyzer/mlflow_backup.db

ALL TRAINING COMPLETE!
